<a href="https://colab.research.google.com/github/shanikairoshi/DeepUnfolding-based-FL/blob/main/main_inital_DUQFL_Scaffold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %%capture
!pip install genomic-benchmarks
!pip install qiskit qiskit_machine_learning qiskit_algorithms qiskit-aer




In [2]:
import sys
from pathlib import Path
PROJ = Path.cwd() / "InitialDQFL_Project"
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))
import sys
sys.path.append('/content/drive/MyDrive/InitialDQFL_Project')
# ─── 5. Assemble filenames for each artifact ─────────────────────────────────
drive_root = "/content/drive/MyDrive/InitialDQFL_Project"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import logging

logging.getLogger("qiskit_machine_learning").setLevel(logging.ERROR)
logging.getLogger("qiskit_machine_learning.neural_networks.sampler_qnn").setLevel(logging.ERROR)

Load and Split data

run federated loop and plot

In [13]:
import os
import json
from datetime import datetime
from common.imports import *
from configs.dataset_genome import *     # swap to other configs as needed
#from io_utils.naming import stamp_now, flags, build_param_str, make_filenames
from configs.base_config import *
#from io_utils.naming import build_param_str, stamp_now, make_filenames
from training.data_factory import *
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from training.scaffold_utils import *
from training.metrics import *
from training.loop import *
from training.fedprox_utils import *

# ---------------- method selection ----------------
method_name = "scaffold"   # "duqfl" | "fedprox" | "scaffold" | "fedavg_fixed_spsa" | "fedavg_tuned_adam"

fedprox_mu = 1e-2
scaffold_lr = 0.1
local_maxiter = 25

# ---------------- build data ----------------
clients, test_sequences, test_labels, num_features = build_clients_and_meta(
    dataset_name=dataset_name,
    split_type=split_type,
    num_clients=num_clients,
    num_epochs=num_epochs,
    samples_per_epoch=samples_per_epoch,
    word_size=word_size,
    global_seed=global_seed,
    mnist_n_features=mnist_n_features,
    mnist_digit_a=mnist_digit_a,
    mnist_digit_b=mnist_digit_b,
    breast_pca_n_features=breast_pca_n_features,
    non_iid_ratio=non_iid_ratio,
    quantity_variation=quantity_variation,
    noniid_seed=noniid_seed,
)

X_val, y_val = test_sequences, test_labels

# ---------------- run folder ----------------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_name}_{split_type}_{stamp}"
method_root = os.path.join(drive_root, "results", method_name, run_name)
os.makedirs(method_root, exist_ok=True)

# ---------------- method state ----------------
scaffold_state = None
param_dim = len(real_amplitudes(num_qubits=len(test_sequences[0]), reps=3).parameters)

if method_name == "scaffold":
    scaffold_state = init_scaffold_state(
        num_clients=len(clients),
        param_dim=param_dim,
    )

extra_config = {
    "method_name": method_name,
    "fedprox_mu": fedprox_mu if method_name == "fedprox" else None,
    "scaffold_state": scaffold_state if method_name == "scaffold" else None,
    "scaffold_lr": scaffold_lr if method_name == "scaffold" else None,
    "local_maxiter": local_maxiter,
}

# ---------------- output files ----------------
global_csv_file = os.path.join(method_root, "global_accuracies.csv")
validation_csv_file = os.path.join(method_root, "validation.csv")
round_metrics_csv_file = os.path.join(method_root, "round_metrics.csv")
client_trace_csv_file = os.path.join(method_root, "client_unfold_trace.csv")
subset_csv_file = None if method_name != "duqfl" else os.path.join(method_root, "subset_selection.csv")
outer_meta_csv_file = None if method_name != "duqfl" else os.path.join(method_root, "outer_meta.csv")
global_params_npz_file = os.path.join(method_root, "global_params.npz")

# ---------------- config record ----------------
run_config = {
    "method_name": method_name,
    "dataset_name": dataset_name,
    "split_type": split_type,
    "num_clients": num_clients,
    "num_federated_layers": num_federated_layers,
    "num_deep_unfolding_iterations": num_deep_unfolding_iterations,
    "initial_learning_rate": initial_learning_rate,
    "initial_perturbation": initial_perturbation,
    "fedprox_mu": fedprox_mu if method_name == "fedprox" else None,
    "scaffold_lr": scaffold_lr if method_name == "scaffold" else None,
    "local_maxiter": local_maxiter,
    "global_seed": global_seed,
    "results_dir": method_root,
    "global_csv_file": global_csv_file,
    "validation_csv_file": validation_csv_file,
    "round_metrics_csv_file": round_metrics_csv_file,
    "client_trace_csv_file": client_trace_csv_file,
    "subset_csv_file": subset_csv_file,
    "outer_meta_csv_file": outer_meta_csv_file,
    "global_params_npz_file": global_params_npz_file,
}

with open(os.path.join(method_root, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

def print_run_config(run_config, method_root):
    print("\n" + "="*70)
    print("CURRENT RUN CONFIGURATION")
    print("="*70)
    for k, v in run_config.items():
        print(f"{k}: {v}")
    print("="*70 + "\n")

print_run_config(run_config, method_root)

from ml.meta_controller import MetaConfig
meta_cfg = MetaConfig()

# ---------------- run ----------------
metrics = metrics_init(log_path=round_metrics_csv_file)

global_acc, clients_train, clients_test, round_times, val_losses, info_last = run_federated_training_meta(
    clients=clients,
    num_federated_layers=num_federated_layers,
    num_deep_unfolding_iterations=num_deep_unfolding_iterations,
    initial_learning_rate=initial_learning_rate,
    initial_perturbation=initial_perturbation,
    num_features=num_features,
    global_csv_file=global_csv_file,
    validation_csv_file=validation_csv_file,
    test_sequences=test_sequences,
    test_labels=test_labels,
    X_val=X_val,
    y_val=y_val,
    metrics=metrics,
    use_deep_unfolding=use_deep_unfolding,
    use_outer_meta=use_outer_meta,
    meta_cfg=meta_cfg,
    global_seed=global_seed,
    client_trace_csv_file=client_trace_csv_file,
    subset_csv_file=subset_csv_file,
    outer_meta_csv_file=outer_meta_csv_file,
    global_params_npz_file=global_params_npz_file,
    extra_config=extra_config,
)


CURRENT RUN CONFIGURATION
method_name: scaffold
dataset_name: Genome
split_type: NONIID
num_clients: 10
num_federated_layers: 10
num_deep_unfolding_iterations: 5
initial_learning_rate: 0.13
initial_perturbation: 0.13
fedprox_mu: None
scaffold_lr: 0.1
local_maxiter: 25
global_seed: 42
results_dir: /content/drive/MyDrive/InitialDQFL_Project/Results/results/scaffold/Genome_NONIID_20260502_030335
global_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/scaffold/Genome_NONIID_20260502_030335/global_accuracies.csv
validation_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/scaffold/Genome_NONIID_20260502_030335/validation.csv
round_metrics_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/scaffold/Genome_NONIID_20260502_030335/round_metrics.csv
client_trace_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/scaffold/Genome_NONIID_20260502_030335/client_unfold_trace.csv
subset_csv_file: None
outer_meta_csv_file: None

Training Progress:   0%|          | 0/10 [00:00<?, ?it/s]

  [Round 0 | Client 0] train_acc=0.7755, test_acc=0.5060, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 1] train_acc=0.7895, test_acc=0.5380, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 2] train_acc=0.8846, test_acc=0.5420, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 3] train_acc=0.7971, test_acc=0.4910, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 4] train_acc=0.8169, test_acc=0.5100, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 5] train_acc=0.7556, test_acc=0.4890, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 0 | Client 6] train_acc=0.7619, test_acc=0.5070, final_lr=0.130000, final

Training Progress:  10%|█         | 1/10 [25:59<3:53:57, 1559.74s/it]

[Round   0] acc_g=0.519 (μ=0.514, σ=0.017, FG=0.048) | t=1539.498s, val=0.793, meta=0.793 | subset=soft_cluster, eff_n=2.989, top1=0.356
[Round 0] global_acc=0.5190 val_loss=0.7935 meta_loss=0.7935
  [Round 1 | Client 0] train_acc=0.8980, test_acc=0.6590, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 1 | Client 1] train_acc=0.7895, test_acc=0.4890, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 1 | Client 2] train_acc=0.8857, test_acc=0.6210, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 1 | Client 3] train_acc=0.7969, test_acc=0.4900, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 1 | Client 4] train_acc=0.8462, test_acc=0.7600, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 1 | Client 5] train_acc=0.79

...

[Round   7] acc_g=0.715 (μ=0.626, σ=0.084, FG=0.207) | t=1551.376s, val=0.536, meta=0.536 | subset=soft_cluster, eff_n=2.747, top1=0.461
[Round 7] global_acc=0.7150 val_loss=0.5359 meta_loss=0.5359
  [Round 8 | Client 0] train_acc=0.8857, test_acc=0.6720, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 8 | Client 1] train_acc=0.8308, test_acc=0.6440, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 8 | Client 2] train_acc=0.9268, test_acc=0.6370, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 8 | Client 3] train_acc=0.8974, test_acc=0.7540, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 8 | Client 4] train_acc=0.8478, test_acc=0.5650, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 8 | Client 5] train_acc=0.80

Training Progress:  90%|█████████ | 9/10 [3:50:51<25:07, 1507.03s/it]

[Round   8] acc_g=0.756 (μ=0.654, σ=0.064, FG=0.186) | t=1428.721s, val=0.506, meta=0.506 | subset=soft_cluster, eff_n=2.135, top1=0.627
[Round 8] global_acc=0.7560 val_loss=0.5064 meta_loss=0.5064
  [Round 9 | Client 0] train_acc=0.8333, test_acc=0.5740, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 9 | Client 1] train_acc=0.8158, test_acc=0.5690, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 9 | Client 2] train_acc=0.8571, test_acc=0.5970, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 9 | Client 3] train_acc=0.9444, test_acc=0.7430, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 9 | Client 4] train_acc=0.8500, test_acc=0.6870, final_lr=0.130000, final_pert=0.130000, loss_last=0.000000, loss_delta=0.000000, accept=0.0000
  [Round 9 | Client 5] train_acc=0.87

Training Progress: 100%|██████████| 10/10 [4:10:39<00:00, 1503.93s/it]

[Round   9] acc_g=0.728 (μ=0.637, σ=0.069, FG=0.171) | t=1167.092s, val=0.518, meta=0.518 | subset=soft_cluster, eff_n=2.298, top1=0.585
[Round 9] global_acc=0.7280 val_loss=0.5175 meta_loss=0.5175
